In [9]:
import pandas as pd
import numpy as np
import re
import joblib
import warnings
from collections import defaultdict

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
import lightgbm as lgb

warnings.filterwarnings("ignore")
print("Imports ready")

Imports ready


In [10]:
comments = pd.read_csv("/kaggle/input/datasets/jyotishmankumarrr/cleaned-set/phase1_clean_comments_v3.csv")
labeled = pd.read_csv("/kaggle/input/datasets/jyotishmankumarrr/labelled-datasets/combined_labeled_comments (1).csv")

comments["comment_text"] = comments["comment_text"].astype(str)
labeled["comment_text"] = labeled["comment_text"].astype(str)
labeled["label"] = labeled["label"].astype(int)

print("Comments:", len(comments))
print("Labeled:", len(labeled))
print("Label distribution:")
print(labeled["label"].value_counts())

Comments: 31920
Labeled: 32205
Label distribution:
label
0    31920
1      285
Name: count, dtype: int64


In [11]:
df = labeled.dropna(subset=["comment_text", "label"]).copy()
df = df[df["comment_text"].str.strip() != ""]

X = df["comment_text"]
y = df["label"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=48,
    class_weight="balanced",
    random_state=42,
    verbose=-1
)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_val_tfidf)
print(classification_report(y_val, y_pred, target_names=["Genuine", "Bot"]))
print(confusion_matrix(y_val, y_pred))

joblib.dump(model, "/kaggle/working/bot_text_model_v2.pkl")
joblib.dump(tfidf, "/kaggle/working/tfidf_vectorizer_v2.pkl")
print("Saved text model")

              precision    recall  f1-score   support

     Genuine       0.99      1.00      1.00      7981
         Bot       0.94      0.24      0.38        71

    accuracy                           0.99      8052
   macro avg       0.97      0.62      0.69      8052
weighted avg       0.99      0.99      0.99      8052

[[7980    1]
 [  54   17]]
Saved text model


In [12]:
comments["has_url"] = comments["comment_text"].str.contains(
    r"http|www\.|t\.me|wa\.me|bit\.ly", case=False, regex=True
).astype(int)

def unique_ratio(series):
    series = series.fillna("").astype(str)
    return series.nunique() / max(len(series), 1)

account_stats = comments.groupby("author_channel_id").agg(
    total_comments=("comment_text", "count"),
    videos_commented=("video_id", "nunique"),
    unique_text_ratio=("comment_text", unique_ratio),
    url_ratio=("has_url", "mean"),
).reset_index()

def account_risk(row):
    score = 0.0
    if row["videos_commented"] >= 3:
        score += 0.40
    elif row["videos_commented"] == 2:
        score += 0.20
    if row["total_comments"] >= 10:
        score += 0.25
    elif row["total_comments"] >= 5:
        score += 0.15
    if row["unique_text_ratio"] <= 0.40:
        score += 0.25
    elif row["unique_text_ratio"] <= 0.70:
        score += 0.10
    if row["url_ratio"] >= 0.40:
        score += 0.20
    return min(score, 1.0)

account_stats["account_risk_score"] = account_stats.apply(account_risk, axis=1)

df = comments.merge(
    account_stats[["author_channel_id", "account_risk_score"]],
    on="author_channel_id",
    how="left"
)
df["account_risk_score"] = df["account_risk_score"].fillna(0.0)

print(account_stats["account_risk_score"].describe())
print("Accounts with risk >= 0.6:", (account_stats["account_risk_score"] >= 0.6).sum())

count    26694.000000
mean         0.004576
std          0.031564
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.700000
Name: account_risk_score, dtype: float64
Accounts with risk >= 0.6: 2


In [13]:
def spam_pattern_score(text):
    t = str(text).lower()
    score = 0.0
    if re.search(r"(http|www\.|t\.me|wa\.me|bit\.ly|tinyurl)", t):
        score += 0.20
    if re.search(r"(whatsapp|telegram|dm me|inbox me|contact me|free recovery|lost crypto|signal group)", t):
        score += 0.30
    if re.search(r"(𝟣|𝟤|𝟥|𝟦|𝟧|𝟨|𝟩|𝟪|𝟫|𝟢)", str(text)) and re.search(r"(thanks|contact|chat|pal|dm|inbox)", t):
        score += 0.30
    return min(score, 0.50)

df["spam_pattern_score"] = df["comment_text"].apply(spam_pattern_score)

def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text

df["norm_text"] = df["comment_text"].map(normalize_text)
df["text_len"] = df["comment_text"].str.len()

work = df[df["text_len"] >= 30].copy()
grouped = work.groupby("norm_text").agg(
    unique_accounts=("author_channel_id", "nunique"),
    comment_count=("comment_text", "count")
).reset_index()

weak_set = set(
    grouped.loc[
        (grouped["unique_accounts"] >= 2) & (grouped["comment_count"] >= 2),
        "norm_text"
    ].astype(str)
)

df["coordination_score"] = df["norm_text"].isin(weak_set).astype(float) * 0.20

print("Spam > 0:", (df["spam_pattern_score"] > 0).sum())
print("Coordination flagged:", (df["coordination_score"] > 0).sum())

Spam > 0: 194
Coordination flagged: 491


In [14]:
model = joblib.load("/kaggle/working/bot_text_model_v2.pkl")
tfidf = joblib.load("/kaggle/working/tfidf_vectorizer_v2.pkl")

X_all = tfidf.transform(df["comment_text"].astype(str).tolist())
df["text_bot_prob"] = model.predict_proba(X_all)[:, 1]

def text_suspicion(prob):
    if prob >= 0.85:
        return "strong"
    if prob >= 0.60:
        return "moderate"
    if prob >= 0.40:
        return "weak"
    return "none"

df["text_suspicion"] = df["text_bot_prob"].apply(text_suspicion)
print(df["text_suspicion"].value_counts())

text_suspicion
none    31919
weak        1
Name: count, dtype: int64


In [15]:
def hard_scam_flag(text):
    t = str(text).lower()
    if re.search(r"(free recovery|lost crypto|signal group|guaranteed profit)", t):
        return 1
    if re.search(r"(whatsapp|telegram|wa\.me|t\.me|bit\.ly|tinyurl)", t):
        return 1
    if re.search(r"\b(dm me|inbox me)\b", t):
        return 1
    if re.search(r"\bcontact me\b", t) and re.search(r"(whatsapp|telegram|number|phone|profit|signal|recover|thanks|pal)", t):
        return 1
    if re.search(r"(𝟣|𝟤|𝟥|𝟦|𝟧|𝟨|𝟩|𝟪|𝟫|𝟢)", str(text)) and re.search(r"(thanks|contact|chat|whatsapp|telegram|pal|dm|inbox)", t):
        return 1
    return 0

df["hard_scam"] = df["comment_text"].apply(hard_scam_flag)

df["structural_support"] = (
    (df["account_risk_score"] >= 0.30) |
    (df["spam_pattern_score"] >= 0.20) |
    (df["coordination_score"] >= 0.20)
).astype(int)

def decide(row):
    if row["hard_scam"] == 1:
        return "High", 0.90, "Hard scam/contact pattern"
    if row["account_risk_score"] >= 0.60:
        return "High", 0.85, "Strong account farm risk"
    if row["coordination_score"] >= 0.60:
        return "High", 0.80, "Strong multi-account coordination"
    if row["text_suspicion"] == "strong" and row["structural_support"] == 1:
        return "High", 0.75, "Strong text suspicion + structural support"
    if row["text_suspicion"] in ["strong", "moderate"]:
        return "Medium", 0.45, "Text suspicion without enough support"
    if row["structural_support"] == 1:
        return "Medium", 0.35, "Weak/moderate structural signals"
    return "Low", 0.05, "No strong bot evidence"

levels, scores, reasons = [], [], []
for _, row in df.iterrows():
    level, score, reason = decide(row)
    levels.append(level)
    scores.append(score)
    reasons.append(reason)

df["final_risk_level"] = levels
df["final_risk_score"] = scores
df["final_reasons"] = reasons

print(df["final_risk_level"].value_counts())
df.to_csv("/kaggle/working/comments_final_hardened.csv", index=False)
print("Saved: comments_final_hardened.csv")

final_risk_level
Low       30786
Medium     1060
High         74
Name: count, dtype: int64
Saved: comments_final_hardened.csv


In [16]:
def video_integrity_report(data, video_name=None, top_n=8):
    d = data.copy()
    if video_name is not None:
        d = d[d["video_name"] == video_name]
        title = video_name
    else:
        title = "All Comments"

    total = len(d)
    low = (d["final_risk_level"] == "Low").sum()
    medium = (d["final_risk_level"] == "Medium").sum()
    high = (d["final_risk_level"] == "High").sum()

    pct_low = round(100 * low / total, 2)
    pct_med = round(100 * medium / total, 2)
    pct_high = round(100 * high / total, 2)
    authenticity = round((low * 1.0 + medium * 0.5 + high * 0.0) / total * 100, 2)
    video_risk = "Low" if authenticity >= 85 else ("Medium" if authenticity >= 60 else "High")

    print("=" * 70)
    print("YOUTUBE COMMENT INTEGRITY REPORT")
    print("=" * 70)
    print(f"Video: {title}")
    print(f"Total comments analyzed: {total}")
    print("-" * 70)
    print(f"Genuine (Low):       {low:5d} ({pct_low}%)")
    print(f"Suspicious (Medium): {medium:5d} ({pct_med}%)")
    print(f"Bot/Spam (High):     {high:5d} ({pct_high}%)")
    print("-" * 70)
    print(f"Overall Authenticity Score: {authenticity} / 100")
    print(f"Video Risk Level: {video_risk}")
    print("-" * 70)

    print("\nTop suspicious comments:")
    top = d.sort_values("final_risk_score", ascending=False).head(top_n)
    for _, row in top.iterrows():
        print(f"\n[{row['final_risk_level']}] {row['final_risk_score']}")
        print(str(row["comment_text"])[:200])
        print("Reason:", row["final_reasons"])

    print("\nTop suspicious accounts:")
    acc = (
        d[d["final_risk_level"].isin(["High", "Medium"])]
        .groupby("author_channel_id")
        .agg(
            comments=("comment_text", "count"),
            high=("final_risk_level", lambda x: (x == "High").sum()),
            avg_score=("final_risk_score", "mean"),
            max_account_risk=("account_risk_score", "max")
        )
        .sort_values(["high", "avg_score"], ascending=False)
        .head(8)
    )
    print(acc)

# Full dataset report
video_integrity_report(df)

# One sample video
sample_video = df["video_name"].value_counts().index[0]
print("\n\n")
video_integrity_report(df, video_name=sample_video)

YOUTUBE COMMENT INTEGRITY REPORT
Video: All Comments
Total comments analyzed: 31920
----------------------------------------------------------------------
Genuine (Low):       30786 (96.45%)
Suspicious (Medium):  1060 (3.32%)
Bot/Spam (High):        74 (0.23%)
----------------------------------------------------------------------
Overall Authenticity Score: 98.11 / 100
Video Risk Level: Low
----------------------------------------------------------------------

Top suspicious comments:

[High] 0.9
𝖶𝗁𝖺𝗍𝗌𝖠𝗉𝗉𝖬𝖤┼𝟣𝟥𝟢𝟧𝟦𝟤𝟩𝟢𝟫𝟢𝟨👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻大不🙏🙏ፈthanks for taking the time,look foward to hearing from you...
Reason: Hard scam/contact pattern

[High] 0.9
Ꮯ𝗈𝗇𝗍𝖺𝖼𝗍𝖬𝖤┼𝟣𝟥𝟢𝟧𝟦𝟤𝟩𝟢𝟫𝟢𝟨👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻👍🏻大不🙏🙏ፈthanks for taking the time,look foward to hearing from you...
Reason: Hard scam/contact pattern

[High] 0.9
Ꮯ𝗈𝗇𝗍𝖺𝖼𝗍𝖬E┼𝟣𝟥𝟢𝟧𝟦𝟤𝟩𝟢𝟫𝟢𝟨👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼监控不是传感器🙏🙏Big thanks pal.
Reason: Hard scam/contact pattern

[High] 0.9
Ꮯ𝗈𝗇𝗍𝖺𝖼𝗍𝖬E┼𝟣𝟥𝟢𝟧𝟦𝟤𝟩𝟢𝟫𝟢𝟨👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼监控不是传感器🙏🙏Big thanks pal.
Reason: Ha

In [17]:
import re

def cleaned_decide(row):
    text = str(row["comment_text"])
    t = text.lower().strip()
    length = len(t)

    # ---- hard scam always High ----
    if row["hard_scam"] == 1:
        return "High", 0.90, "Hard scam/contact pattern"

    if row["account_risk_score"] >= 0.60:
        return "High", 0.85, "Strong account farm risk"

    if row["coordination_score"] >= 0.60:
        return "High", 0.80, "Strong multi-account coordination"

    # ---- force Low for weak/noise comments ----
    # very short comments are usually genuine reactions
    if length < 20 and row["hard_scam"] == 0:
        return "Low", 0.05, "Short comment with no scam signal"

    # emoji-only / symbol-only
    if re.fullmatch(r"[\W_]+", t):
        return "Low", 0.05, "Emoji/symbol-only comment"

    # common genuine short replies
    if t in {"yes", "no", "ok", "lol", "true", "facts", "same", "real", "this"}:
        return "Low", 0.05, "Common genuine short reply"

    # ---- High only with strong support ----
    if row["text_suspicion"] == "strong" and row["structural_support"] == 1:
        return "High", 0.75, "Strong text suspicion + structural support"

    # ---- better Medium rules ----
    # text suspicion alone can be Medium
    if row["text_suspicion"] in ["strong", "moderate"]:
        return "Medium", 0.45, "Text suspicion without enough support"

    # structural alone must be stronger to become Medium
    strong_structural = (
        (row["account_risk_score"] >= 0.45) or
        (row["spam_pattern_score"] >= 0.30) or
        (row["coordination_score"] >= 0.20 and length >= 30)
    )
    if strong_structural:
        return "Medium", 0.35, "Stronger structural signals"

    return "Low", 0.05, "No strong bot evidence"


levels, scores, reasons = [], [], []
for _, row in df.iterrows():
    level, score, reason = cleaned_decide(row)
    levels.append(level)
    scores.append(score)
    reasons.append(reason)

df["final_risk_level"] = levels
df["final_risk_score"] = scores
df["final_reasons"] = reasons

print("CLEANED DISTRIBUTION")
print(df["final_risk_level"].value_counts())

print("\nHigh samples:")
print(
    df[df["final_risk_level"] == "High"][["comment_text", "final_reasons"]]
    .head(8)
    .to_string(index=False)
)

print("\nMedium samples:")
print(
    df[df["final_risk_level"] == "Medium"][["comment_text", "final_reasons"]]
    .head(8)
    .to_string(index=False)
)

df.to_csv("/kaggle/working/comments_final_cleaned.csv", index=False)
print("\nSaved: comments_final_cleaned.csv")

# refresh report
video_integrity_report(df)

CLEANED DISTRIBUTION
final_risk_level
Low       31590
Medium      256
High         74
Name: count, dtype: int64

High samples:
                                                                                      comment_text             final_reasons
                                                                                      Kindly dm me Hard scam/contact pattern
Ꮯ𝗈𝗇𝗍𝖺𝖼𝗍𝖬E┼𝟣𝟥𝟢𝟧𝟦𝟤𝟩𝟢𝟫𝟢𝟨👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼监控不是传感器🙏🙏Big thanks, looking forward to hearing from you. Hard scam/contact pattern
Ꮯ𝗈𝗇𝗍𝖺𝖼𝗍𝖬E┼𝟣𝟥𝟢𝟧𝟦𝟤𝟩𝟢𝟫𝟢𝟨👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼监控不是传感器🙏🙏Big thanks, looking forward to hearing from you. Hard scam/contact pattern
Ꮯ𝗈𝗇𝗍𝖺𝖼𝗍𝖬E┼𝟣𝟥𝟢𝟧𝟦𝟤𝟩𝟢𝟫𝟢𝟨👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼监控不是传感器🙏🙏Big thanks, looking forward to hearing from you. Hard scam/contact pattern
Ꮯ𝗈𝗇𝗍𝖺𝖼𝗍𝖬E┼𝟣𝟥𝟢𝟧𝟦𝟤𝟩𝟢𝟫𝟢𝟨👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼监控不是传感器🙏🙏Big thanks, looking forward to hearing from you. Hard scam/contact pattern
Ꮯ𝗈𝗇𝗍𝖺𝖼𝗍𝖬E┼𝟣𝟥𝟢𝟧𝟦𝟤𝟩𝟢𝟫𝟢𝟨👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼👍🏼监控不是传感器🙏🙏Big thanks, looking forward to hearing from you. Hard scam/contact patter

In [18]:
!pip install -q gradio google-api-python-client

import re
import time
import pandas as pd
import numpy as np
import joblib
import gradio as gr
import matplotlib.pyplot as plt

from urllib.parse import urlparse, parse_qs
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from kaggle_secrets import UserSecretsClient

print("Ready")

Ready


In [19]:
user_secrets = UserSecretsClient()
YOUTUBE_API_KEY = user_secrets.get_secret("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

# load model if available
try:
    text_model = joblib.load("/kaggle/working/bot_text_model_v2.pkl")
    tfidf = joblib.load("/kaggle/working/tfidf_vectorizer_v2.pkl")
    HAS_MODEL = True
except:
    text_model, tfidf, HAS_MODEL = None, None, False

def extract_video_id(url: str):
    url = (url or "").strip()
    if not url:
        return None
    parsed = urlparse(url)
    if parsed.netloc in ("youtu.be", "www.youtu.be"):
        vid = parsed.path.lstrip("/")
        return vid.split("?")[0] if vid else None
    if "youtube.com" in parsed.netloc:
        q = parse_qs(parsed.query)
        if "v" in q:
            return q["v"][0]
        # shorts / embed
        parts = [p for p in parsed.path.split("/") if p]
        if parts and parts[0] in ("shorts", "embed", "live") and len(parts) > 1:
            return parts[1]
    return None

def hard_scam_flag(text):
    t = str(text).lower()
    if re.search(r"(free recovery|lost crypto|signal group|guaranteed profit)", t):
        return 1
    if re.search(r"(whatsapp|telegram|wa\.me|t\.me|bit\.ly|tinyurl)", t):
        return 1
    if re.search(r"\b(dm me|inbox me)\b", t):
        return 1
    if re.search(r"\bcontact me\b", t) and re.search(r"(whatsapp|telegram|number|phone|profit|signal|recover|thanks|pal)", t):
        return 1
    if re.search(r"(𝟣|𝟤|𝟥|𝟦|𝟧|𝟨|𝟩|𝟪|𝟫|𝟢)", str(text)) and re.search(r"(thanks|contact|chat|whatsapp|telegram|pal|dm|inbox)", t):
        return 1
    return 0

def spam_pattern_score(text):
    t = str(text).lower()
    score = 0.0
    if re.search(r"(http|www\.|t\.me|wa\.me|bit\.ly|tinyurl)", t):
        score += 0.20
    if re.search(r"(whatsapp|telegram|dm me|inbox me|contact me|free recovery|lost crypto|signal group)", t):
        score += 0.30
    if re.search(r"(𝟣|𝟤|𝟥|𝟦|𝟧|𝟨|𝟩|𝟪|𝟫|𝟢)", str(text)) and re.search(r"(thanks|contact|chat|pal|dm|inbox)", t):
        score += 0.30
    return min(score, 0.50)

def fetch_comments(video_id, max_comments=1000):
    rows = []
    token = None
    fetched = 0

    while fetched < max_comments:
        try:
            req = youtube.commentThreads().list(
                part="snippet,replies",
                videoId=video_id,
                maxResults=min(100, max_comments - fetched),
                pageToken=token,
                textFormat="plainText"
            )
            resp = req.execute()
        except HttpError as e:
            msg = str(e)
            if "commentsDisabled" in msg:
                raise ValueError("Comments are disabled on this video.")
            if "videoNotFound" in msg:
                raise ValueError("Video not found / invalid link.")
            if "quota" in msg.lower():
                raise ValueError("YouTube API quota exceeded. Try later.")
            raise ValueError(f"YouTube API error: {e}")

        items = resp.get("items", [])
        if not items:
            break

        for item in items:
            top = item["snippet"]["topLevelComment"]["snippet"]
            rows.append({
                "comment_text": top.get("textDisplay", ""),
                "published_at": top.get("publishedAt", ""),
                "like_count": top.get("likeCount", 0),
                "author_channel_id": top.get("authorChannelId", {}).get("value"),
                "is_reply": False
            })
            fetched += 1
            if fetched >= max_comments:
                break

            # include some replies
            for rep in item.get("replies", {}).get("comments", []):
                if fetched >= max_comments:
                    break
                s = rep["snippet"]
                rows.append({
                    "comment_text": s.get("textDisplay", ""),
                    "published_at": s.get("publishedAt", ""),
                    "like_count": s.get("likeCount", 0),
                    "author_channel_id": s.get("authorChannelId", {}).get("value"),
                    "is_reply": True
                })
                fetched += 1

        token = resp.get("nextPageToken")
        if not token:
            break
        time.sleep(0.05)

    return pd.DataFrame(rows)

def score_comments(df):
    df = df.copy()
    df["comment_text"] = df["comment_text"].astype(str)
    df["hard_scam"] = df["comment_text"].map(hard_scam_flag)
    df["spam_pattern_score"] = df["comment_text"].map(spam_pattern_score)

    # account risk quick
    acc = df.groupby("author_channel_id").agg(
        total_comments=("comment_text", "count"),
        unique_text_ratio=("comment_text", lambda s: s.nunique() / max(len(s), 1))
    ).reset_index()

    def acc_risk(r):
        score = 0.0
        if r["total_comments"] >= 10:
            score += 0.35
        elif r["total_comments"] >= 5:
            score += 0.20
        if r["unique_text_ratio"] <= 0.4:
            score += 0.30
        return min(score, 1.0)

    acc["account_risk_score"] = acc.apply(acc_risk, axis=1)
    df = df.merge(acc[["author_channel_id", "account_risk_score"]], on="author_channel_id", how="left")
    df["account_risk_score"] = df["account_risk_score"].fillna(0.0)

    # text model optional
    if HAS_MODEL:
        X = tfidf.transform(df["comment_text"].tolist())
        df["text_bot_prob"] = text_model.predict_proba(X)[:, 1]
    else:
        df["text_bot_prob"] = 0.0

    def text_suspicion(p):
        if p >= 0.85:
            return "strong"
        if p >= 0.60:
            return "moderate"
        if p >= 0.40:
            return "weak"
        return "none"

    df["text_suspicion"] = df["text_bot_prob"].map(text_suspicion)

    def decide(row):
        text = str(row["comment_text"]).strip()
        if row["hard_scam"] == 1:
            return "High", 0.90, "Hard scam/contact pattern"
        if row["account_risk_score"] >= 0.60:
            return "High", 0.85, "Strong account farm risk"
        if len(text) < 20 and row["hard_scam"] == 0:
            return "Low", 0.05, "Short comment with no scam signal"
        if row["text_suspicion"] == "strong" and (row["spam_pattern_score"] >= 0.20 or row["account_risk_score"] >= 0.30):
            return "High", 0.75, "Strong text suspicion + support"
        if row["text_suspicion"] in ["strong", "moderate"]:
            return "Medium", 0.45, "Text suspicion"
        if row["spam_pattern_score"] >= 0.30 or row["account_risk_score"] >= 0.45:
            return "Medium", 0.35, "Structural signals"
        return "Low", 0.05, "No strong bot evidence"

    levels, scores, reasons = [], [], []
    for _, r in df.iterrows():
        level, score, reason = decide(r)
        levels.append(level)
        scores.append(score)
        reasons.append(reason)

    df["final_risk_level"] = levels
    df["final_risk_score"] = scores
    df["final_reasons"] = reasons
    return df

def make_chart(df):
    counts = df["final_risk_level"].value_counts().reindex(["Low", "Medium", "High"]).fillna(0)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(counts.index, counts.values)
    ax.set_title("Comment Risk Distribution")
    ax.set_ylabel("Count")
    plt.tight_layout()
    return fig

In [20]:
def analyze(url, max_comments):
    try:
        max_comments = int(max_comments)
        if max_comments <= 0:
            return "Max comments must be > 0", None, None, None

        video_id = extract_video_id(url)
        if not video_id:
            return "Invalid YouTube link. Paste a full video URL.", None, None, None

        df = fetch_comments(video_id, max_comments=max_comments)
        if df is None or len(df) == 0:
            return "No comments found (or comments unavailable).", None, None, None

        scored = score_comments(df)

        total = len(scored)
        low = (scored["final_risk_level"] == "Low").sum()
        med = (scored["final_risk_level"] == "Medium").sum()
        high = (scored["final_risk_level"] == "High").sum()
        auth = round((low * 1.0 + med * 0.5 + high * 0.0) / total * 100, 2)
        risk = "Low" if auth >= 85 else ("Medium" if auth >= 60 else "High")

        summary = f"""
### YouTube Comment Integrity Report
- Video ID: `{video_id}`
- Comments analyzed: **{total}** (requested {max_comments})
- Genuine (Low): **{low}** ({round(100*low/total,2)}%)
- Suspicious (Medium): **{med}** ({round(100*med/total,2)}%)
- Bot/Spam (High): **{high}** ({round(100*high/total,2)}%)
- Authenticity Score: **{auth}/100**
- Video Risk: **{risk}**
"""

        top = scored.sort_values("final_risk_score", ascending=False)[
            ["comment_text", "final_risk_level", "final_risk_score", "final_reasons"]
        ].head(15)

        chart = make_chart(scored)
        return summary, chart, top, scored

    except Exception as e:
        return f"Error: {str(e)}", None, None, None


with gr.Blocks(title="YouTube Comment Integrity Analyzer") as demo:
    gr.Markdown("# YouTube Comment Integrity Analyzer")
    gr.Markdown("Paste a YouTube link, set max comments, generate integrity report.")

    with gr.Row():
        url = gr.Textbox(label="YouTube URL", placeholder="https://www.youtube.com/watch?v=...")
        max_c = gr.Number(label="Max comments", value=1000, precision=0)

    btn = gr.Button("Analyze", variant="primary")

    out_md = gr.Markdown()
    out_plot = gr.Plot()
    out_table = gr.Dataframe()
    out_full = gr.Dataframe(visible=False)

    btn.click(analyze, inputs=[url, max_c], outputs=[out_md, out_plot, out_table, out_full])

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://0b4a86ed17b1dd2910.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
